# 01 — Fundamentos de Bancos de Dados

**Objetivos:** distinguir dado, informação, SGBD, modelos de armazenamento e critérios de seleção

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia técnico — Fundamentos de bancos de dados](./reference_materials/01_fundamentos_bancos_dados.pdf)
- [Cheat sheet — Fundamentos de bancos de dados](./cheat_sheets/cheat_sheet_fundamentos_bancos_dados.pdf)
- [Documentação e arquivos de resultados](./README.pt-BR.md)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## Dado, informação e persistência

Dado é registro; informação é dado contextualizado. Um SGBD oferece armazenamento, consulta, controle de acesso e regras de integridade. Arquivos CSV são úteis, mas não garantem integridade referencial nem transações entre múltiplas tabelas.

In [2]:
df = pd.read_csv(DATA/'leituras_sensores.csv')
print(df.shape); display(df.head());print(df.dtypes)

(72, 5)


,id_evento,id_sensor,data_hora,temperatura_c,umidade_pct
0,1,2,2025-01-01 08:00:00,22.62,45.5
1,2,3,2025-01-01 09:00:00,23.23,49.5
2,3,1,2025-01-01 10:00:00,21.21,58.5
3,4,2,2025-01-01 11:00:00,23.38,46.7
4,5,3,2025-01-01 12:00:00,23.67,45.6


id_evento          int64
id_sensor          int64
data_hora            str
temperatura_c    float64
umidade_pct      float64
dtype: object


## Comparação dos modelos

**Relacional:** tabelas, esquema, relacionamentos e SQL. **Documental:** documentos aninhados e consultas por campo. **Colunar distribuído:** particionamento orientado ao padrão de acesso. Não existe banco universalmente melhor; escolha por consultas, consistência, escala e operação.

In [3]:
comparacao=pd.DataFrame({'modelo':['Relacional','Documental','Wide-column'],'exemplo':['PostgreSQL','MongoDB','Cassandra'],'forte_em':['integridade e SQL','documentos flexíveis','distribuição e consultas predefinidas']});display(comparacao)

,modelo,exemplo,forte_em
0,Relacional,PostgreSQL,integridade e SQL
1,Documental,MongoDB,documentos flexíveis
2,Wide-column,Cassandra,distribuição e consultas predefinidas


## Exercício resolvido — escolhas de arquitetura

**Problema:** histórico de sensores da casa inteligente com consultas frequentes por data e sensor. **Solução justificada:** começar pelo relacional devido a chaves e consultas analíticas; considerar soluções distribuídas somente após medir volume e requisitos de disponibilidade.

In [4]:
print('Decisão: SQLite para prototipar; PostgreSQL para concorrência e transações de servidor.')

Decisão: SQLite para prototipar; PostgreSQL para concorrência e transações de servidor.


## Erros comuns

Confundir disponibilidade com ausência de erros, NoSQL com ausência de esquema, e escalabilidade com um requisito presente em qualquer projeto.

## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — Persistência, restrições e decisão tecnológica

A seleção de um SGBD depende de *consultas, integridade, volume, concorrência, disponibilidade e equipe*. Armazenar CSV em disco não equivale a controlar transações ou integridade referencial. Compare **OLTP** (operações curtas, concorrentes) e **OLAP** (agregações analíticas); a divisão é de carga de trabalho, não de fornecedor.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [5]:
import sqlite3
with sqlite3.connect(':memory:') as db:
    db.execute('PRAGMA foreign_keys = ON')
    db.executescript('CREATE TABLE pessoas(id INTEGER PRIMARY KEY, nome TEXT NOT NULL); CREATE TABLE pedidos(id INTEGER PRIMARY KEY, pessoa_id INTEGER NOT NULL REFERENCES pessoas(id));')
    db.execute('INSERT INTO pessoas VALUES (1, "Ana")')
    db.execute('INSERT INTO pedidos VALUES (1, 1)')
    try:
        db.execute('INSERT INTO pedidos VALUES (2, 999)')
    except sqlite3.IntegrityError:
        print('Integridade referencial protegida pelo banco.')
    else:
        raise AssertionError('Restrição FK não aplicada')

Integridade referencial protegida pelo banco.


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [6]:
comparacao.to_csv(OUT / '01_comparacao_modelos.csv', index=False)
print('Exportado: outputs/01_comparacao_modelos.csv')

Exportado: outputs/01_comparacao_modelos.csv
